# 01 · Load and Clean

**Goal:** load the raw Olist CSVs into a SQLite database, check data quality,
and build one clean, order-level analysis table for the rest of the project.

Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(CC BY-NC-SA 4.0). Download the CSVs into `data/` before running this notebook.

## 1. Load raw CSVs into SQLite


In [177]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import empiricaldist

In [178]:
DATA_DIR = Path("../data")
DB_PATH = DATA_DIR / "olist.db"

DB_PATH.unlink(missing_ok=True)   # rebuild from scratch
conn = sqlite3.connect(DB_PATH)

csv_paths = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_paths)} CSV files")

Found 9 CSV files


In [179]:
def table_name_from(path: Path) -> str:
    """olist_orders_dataset.csv -> orders"""
    return path.stem.replace("olist_", "").replace("_dataset", "")

for path in csv_paths:
    df = pd.read_csv(path)
    new_name = table_name_from(path)
    globals()[new_name] = df
    df.to_sql(table_name_from(path), conn, if_exists="replace", index=False)

print("Loaded tables:", [table_name_from(p) for p in csv_paths])

Loaded tables: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category', 'products', 'sellers']


### Verify the load
Compare each table's SQL row count with the raw line count of its CSV file.
A mismatch means the file has line breaks *inside* quoted fields.

In [180]:
print(f"{'TABLE':<35} {'SQL ROWS':>10} {'FILE LINES':>11}")
print("")
for path in csv_paths:
    table = table_name_from(path)
    sql_rows = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    with open(path, encoding="utf-8") as f:
        file_lines = sum(1 for _ in f) - 1      # minus header
    flag = "  <-- mismatch" if sql_rows != file_lines else ""
    print(f"{table:<35} {sql_rows:>10,} {file_lines:>11,}{flag}")

TABLE                                 SQL ROWS  FILE LINES

customers                               99,441      99,441
geolocation                          1,000,163   1,000,163
order_items                            112,650     112,650
order_payments                         103,886     103,886
order_reviews                           99,224     104,719  <-- mismatch
orders                                  99,441      99,441
product_category                            71          71
products                                32,951      32,951
sellers                                  3,095       3,095


In [181]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Table in olist.db:")
for table in tables:
    query = f"SELECT count(*) FROM {table[0]};"
    row = cursor.execute(query).fetchone()
    print(f"{table[0]:<35} has {row[0]} rows")

Table in olist.db:
customers                           has 99441 rows
geolocation                         has 1000163 rows
order_items                         has 112650 rows
order_payments                      has 103886 rows
order_reviews                       has 99224 rows
orders                              has 99441 rows
product_category                    has 71 rows
products                            has 32951 rows
sellers                             has 3095 rows


All tables match their files except `order_reviews` (99,224 rows vs. 104,719 lines):
review comments contain line breaks, so one row can span several lines.
The SQL row count is the correct one.

## 2. Data quality checks


### 2.1 Reviews: Overview

`order_reviews` has 99,224 rows but `orders` has 99,441. Before joining,
check whether any order has more than one review (which would duplicate
orders in the join).

In [182]:
query = """
SELECT
    (SELECT COUNT(*) FROM orders) AS total_orders,
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT order_id) AS total_reviewed_orders,
    COUNT(*) - COUNT(DISTINCT order_id) AS gap_review_orders,
    (SELECT COUNT(*) FROM (
        SELECT order_id
        FROM order_reviews
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) AS multiple_review_orders) AS orders_with_mreviews,
    (SELECT COUNT(*) FROM orders) - COUNT(DISTINCT order_id) AS orders_without_reviews
    
FROM order_reviews
"""
reviews_gap = pd.read_sql_query(query, conn)
reviews_gap

,total_orders,total_reviews,total_reviewed_orders,gap_review_orders,orders_with_mreviews,orders_without_reviews
0,99441,99224,98673,551,547,768


List of Orders with multiple reviews

In [183]:
query = """
SELECT 
    order_id, 
    COUNT(*) AS total_reviews
FROM order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY total_reviews DESC;
"""
result = pd.read_sql_query(query, conn)
result.head(5)

,order_id,total_reviews
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2


1. There 768 orders have no review. (0.77%)
2. There are 547 orders have more than 1 review (0.55%) . There are 4 orders have 3 reviews and 543 orders have 2 reviews.

### 2.2 Reviews: Duplicate reviews per order

In [184]:
query = """
WITH multi AS (
    SELECT order_id
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
),
with_prev AS (
    SELECT
        order_id,
        review_score,
        LAG(review_score) OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp
        ) AS prev_score
    FROM order_reviews
    WHERE order_id IN (SELECT order_id FROM multi)
)
SELECT
    COUNT(*)                            AS review_pairs,
    SUM(review_score <  prev_score)     AS lower,
    SUM(review_score =  prev_score)     AS same,
    SUM(review_score >  prev_score)     AS higher,
    ROUND(AVG(review_score - prev_score), 2) AS avg_change
FROM with_prev
WHERE prev_score IS NOT NULL;

"""
result = pd.read_sql_query(query, conn)
result


,review_pairs,lower,same,higher,avg_change
0,551,114,348,89,-0.15


Same score review is the most portion in 551 review pairs

 **Finding:** 
1. There 768 orders have no review. (0.77%)
2. There are 547 orders have more than 1 review (0.55%) . There are 4 orders have 3 reviews and the rest has 2 reviews.

|  | total_orders | total_reviews | total_reviewed_orders | gap_review_orders | orders_with_mreviews | orders_without_reviews |
| --- | --- | --- | --- | --- | --- | --- |
| **0** | 99441 | 99224 | 98673 | 551 | 547 | 768 |



3. Out of 551 consecutive review pairs, 348 (63%) kept the same score,
114 (21%) dropped and 89 (16%) rose (average change −0.15 points).
Most duplicate reviews agree; when they differ, drops are slightly more common.

|  | review_pairs | lower | same | higher | avg_change |
| --- | --- | --- | --- | --- | --- |
| **0** | 551 | 114 | 348 | 89 | -0.15 |


**Decision:** 

Keep only the **latest** review per order (by `review_answer_timestamp`)
as the customer's final opinion. This affects fewer than 0.3% of orders, so the
choice does not materially change later results. Orders without a review are kept
for delivery analysis but excluded from review-score tests (H1).

In [185]:
query = """
With rank as (
    select *, row_number() over (partition by order_id order by review_answer_timestamp desc) as row_num
    from order_reviews)
select * from rank where row_num =1

"""
new_order_reviews = pd.read_sql_query(query, conn)
new_order_reviews.to_sql('new_order_reviews', conn, if_exists='replace', index=False)
new_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,row_num
0,97ca439bc427b48bc1cd7177abe71365,00010242fe8c5a6d1ba2dd792cb16214,5,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03,1
1,7b07bacd811c4117b742569b04ce3580,00018f77f2f0320c557190d7a144bdd3,4,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13,1
2,0c5b33dea94867d1ac402749e5438e8b,000229ec398224ef6ca0657da4fc703e,5,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31,1
3,f4028d019cb58564807486a6aaf33817,00024acbcdf0a6daa1e931b038114c75,4,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01,1
4,940144190dcba6351888cafa43f3a3a5,00042b26cf59d7ce69dfabb4e55b4fd9,5,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59,1


### 2.3 Orders: Status breakdown & Missing dates

In [193]:
print(orders.info())
orders.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [194]:
query = """
SELECT order_status, count(*) as total_order,
    SUM(order_approved_at IS NULL) as not_approved,
    SUM(order_delivered_carrier_date IS NULL) as missing_carrier,
    SUM(order_delivered_customer_date IS NULL) as missing_customer
FROM orders
GROUP BY order_status
order by total_order desc

"""
order_status = pd.read_sql_query(query, conn)
order_status

,order_status,total_order,not_approved,missing_carrier,missing_customer
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


In [ ]:
new_orders = orders[orders['order_status']=='delivered']
new_orders = new_orders.dropna(subset=['order_approved_at','order_delivered_customer_date','order_delivered_carrier_date']).reset_index(drop=True)
new_orders.to_sql("new_orders", conn, if_exists="replace", index=False)

new_orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 96455 entries, 0 to 96454
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       96455 non-null  str  
 1   customer_id                    96455 non-null  str  
 2   order_status                   96455 non-null  str  
 3   order_purchase_timestamp       96455 non-null  str  
 4   order_approved_at              96455 non-null  str  
 5   order_delivered_carrier_date   96455 non-null  str  
 6   order_delivered_customer_date  96455 non-null  str  
 7   order_estimated_delivery_date  96455 non-null  str  
dtypes: str(8)
memory usage: 5.9 MB


Keep only orders with status `deliveried` and all the dates are not empty

### 2.4 Orders: Timestamp consistency

In [196]:
query = """
SELECT
    COUNT(*) AS n_orders,

    SUM(order_approved_at            < order_purchase_timestamp)      AS approved_before_purchase,
    SUM(order_delivered_carrier_date < order_approved_at)             AS carrier_before_approved,
    SUM(order_delivered_customer_date < order_delivered_carrier_date) AS delivered_before_carrier,
    SUM(order_delivered_customer_date < order_purchase_timestamp)     AS delivered_before_purchase
FROM new_orders;
"""
order_anomalies = pd.read_sql_query(query, conn)
order_anomalies

,n_orders,approved_before_purchase,carrier_before_approved,delivered_before_carrier,delivered_before_purchase
0,96455,0,1350,23,0


There 1373 orders have timestamp not in order.

In [197]:
# Parse timestamp columns (you'll need real datetimes for delivery_days later anyway)
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
new_orders[date_cols] = new_orders[date_cols].apply(pd.to_datetime)

# Rows where the stages are out of order
bad_stages = (
    (new_orders["order_delivered_carrier_date"] < new_orders["order_approved_at"])
    | (new_orders["order_delivered_customer_date"] < new_orders["order_delivered_carrier_date"])
)

print(f"Orders before:  {len(new_orders):,}")
print(f"Dropped:        {bad_stages.sum():,} ({bad_stages.mean():.1%})")

new_orders = new_orders[~bad_stages].copy()
print(f"Orders after:   {len(new_orders):,}")

Orders before:  96,455
Dropped:        1,373 (1.4%)
Orders after:   95,082


In [198]:
new_orders.to_sql("new_orders_clean", conn, if_exists="replace", index=False)

95082

**Findings**
- 96,478 orders (97.0%) are `delivered`; 23 of them are missing at least one
  timestamp (approval, carrier handoff, or customer delivery).
- 6 `canceled` orders have a delivery date (likely returns or data errors).
- 1,107 orders (1.1%) are `shipped` but never received a delivery date.
- Among delivered orders with all timestamps, none was delivered before purchase,
  but `1,373` (≈1.4%) have stages out of order (mostly handed to the carrier
  before payment approval).

**Decision:** 
- Keep `delivered` orders with all three timestamps present and stages
in chronological order: `95082` orders (`97`% of all 99,441 orders),
saved as `new_orders`. This keeps every stage duration (seller handling,
carrier transit) non-negative.

**Limitations**
- Excluded `shipped` orders are likely the slowest deliveries (right-censoring),
  so delivery times and late rates may be slightly understated.
- Orders dropped for out-of-order stages still had valid end-to-end delivery
  times; dropping them is a conservative choice for the stage analysis.

### 2.5 Order items: items and sellers per

`order_items` has one row per item. Before aggregating to one row per order,
check how many orders have multiple items and how many have multiple sellers.

In [216]:
query = """
SELECT n_items, count(*) AS n_orders,  avg_freight
FROM (
    SELECT order_id, MAX(order_item_id) AS n_items, round(avg(freight_value), 2) AS avg_freight
    FROM order_items
    GROUP BY order_id    
)
GROUP BY n_items
ORDER BY n_items;



"""
result = pd.read_sql_query(query, conn)
result

,n_items,n_orders,avg_freight
0,1,88863,13.29
1,2,7516,13.37
2,3,1322,15.10
3,4,505,8.40
4,5,204,11.85
5,6,198,30.44
6,7,22,9.86
7,8,8,28.01
8,9,3,50.97
9,10,8,9.98


In [201]:
query = """
WITH per_order AS (
    SELECT order_id,
           COUNT(*)               AS n_items,
           COUNT(DISTINCT seller_id) AS n_sellers
    FROM order_items
    GROUP BY order_id
)
SELECT n_sellers,
       COUNT(*) AS n_orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM per_order
GROUP BY n_sellers
ORDER BY n_sellers;
"""
result = pd.read_sql_query(query, conn)
result

,n_sellers,n_orders,pct
0,1,97388,98.70
1,2,1219,1.24
2,3,54,0.05
3,4,3,0.00
4,5,2,0.00


**Findings**
- 98,666 orders have items (775 orders in `orders` have none).
- 9,803 orders (9.9%) contain more than one item, so item-level columns
  (freight, weight) must be aggregated to one row per order.
- 1,278 orders (1.3%) have more than one seller, so `seller_state` is ambiguous
  for them.

**Decision:** keep all rows in `order_items`. In the join, items are aggregated
to one row per order (item count, total freight, total weight).

For the 1.3% of orders with more than one seller, `seller_state` is set to NULL; these orders
are kept for delivery-time analysis and excluded only from seller-level analysis.